In [3]:
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 10.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 37.2 MB/s eta 0:00:0000:0100:01


In [4]:
import json
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [6]:
# load JSONL
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

raw = load_jsonl("data/raw/traces.jsonl")
print(f"Loaded {len(raw)} traces")

Loaded 577 traces


In [7]:
# inspect one sample
print(json.dumps(raw[0], indent=2))

{
  "problem": "What is the degree of the polynomial $(4 +5x^3 +100 +2\\pi x^4 + \\sqrt{10}x^4 +9)$?",
  "trace": "The given polynomial is \\(4 + 5x^3 + 100 + 2\\pi x^4 + \\sqrt{10}x^4 + 9\\). To find its degree:\n\n1. **Identify each term and their exponents:**\n   - Constants (4, 100, 9): Exponent is 0.\n   - \\(5x^3\\): Exponent is 3.\n   - \\(2\\pi x^4\\) and \\(\\sqrt{10}x^4\\): Both have exponent 4.\n\n2. **Determine the highest exponent:**\n   The exponents are 0, 3, and 4. The highest is 4.\n\nThus, the degree of the polynomial is 4.\n\n<answer>4</answer>",
  "answer": "4",
  "level": "Level 3",
  "type": "Algebra",
  "tokens": 740,
  "elapsed_sec": 8.63,
  "tokens_per_sec": 75.4
}


In [8]:
# format into chat template
def format_example(item):
    """
    Formats a raw trace into a single training string.
    
    Structure:
        <|user|>       — the math problem
        <|assistant|>  — the full CoT trace including <think> and <answer> tags
    
    This is the format SFTTrainer expects when using dataset_text_field='text'
    """

    return {
        "text": (
            f"<|user|>\n{item['problem']}\n"
            f"<|assistant|>\n{item['trace']}"
        ),
        "level": item['level'],
        "type": item['type']
    }

formatted = [format_example(r) for r in raw]
print(f"Formatted {len(formatted)} examples")

Formatted 577 examples


In [12]:
formatted[0]

{'text': '<|user|>\nWhat is the degree of the polynomial $(4 +5x^3 +100 +2\\pi x^4 + \\sqrt{10}x^4 +9)$?\n<|assistant|>\nThe given polynomial is \\(4 + 5x^3 + 100 + 2\\pi x^4 + \\sqrt{10}x^4 + 9\\). To find its degree:\n\n1. **Identify each term and their exponents:**\n   - Constants (4, 100, 9): Exponent is 0.\n   - \\(5x^3\\): Exponent is 3.\n   - \\(2\\pi x^4\\) and \\(\\sqrt{10}x^4\\): Both have exponent 4.\n\n2. **Determine the highest exponent:**\n   The exponents are 0, 3, and 4. The highest is 4.\n\nThus, the degree of the polynomial is 4.\n\n<answer>4</answer>',
 'level': 'Level 3',
 'type': 'Algebra'}

In [9]:
# inspect one formatted example
print(formatted[0]['text'][:500])

<|user|>
What is the degree of the polynomial $(4 +5x^3 +100 +2\pi x^4 + \sqrt{10}x^4 +9)$?
<|assistant|>
The given polynomial is \(4 + 5x^3 + 100 + 2\pi x^4 + \sqrt{10}x^4 + 9\). To find its degree:

1. **Identify each term and their exponents:**
   - Constants (4, 100, 9): Exponent is 0.
   - \(5x^3\): Exponent is 3.
   - \(2\pi x^4\) and \(\sqrt{10}x^4\): Both have exponent 4.

2. **Determine the highest exponent:**
   The exponents are 0, 3, and 4. The highest is 4.

Thus, the degree of the 


In [15]:
# split into train/val
train_data, val_data = train_test_split(formatted, test_size=0.05, random_state=42)
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

Train: 548 | Val: 29


In [16]:
# convert to HF Dataset and save
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)


In [18]:
train_ds.save_to_disk("data/processed/train")
val_ds.save_to_disk("data/processed/val")

Saving the dataset (0/1 shards):   0%|          | 0/548 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/29 [00:00<?, ? examples/s]

In [19]:
# sanity check
print(train_ds)
print(val_ds)
print(f"\nSample text field:\n{train_ds[0]['text'][:300]}")

Dataset({
    features: ['text', 'level', 'type'],
    num_rows: 548
})
Dataset({
    features: ['text', 'level', 'type'],
    num_rows: 29
})

Sample text field:
<|user|>
Simplify $2a(2a^2 + a) - a^2$.
<|assistant|>
Step-by-step explanation:

We start with the expression:
\[2a(2a^2 + a) - a^2\]

First, distribute the \(2a\) across the terms inside the parentheses:
\[
2a \cdot 2a^2 = 4a^3
\]
\[
2a \cdot a = 2a^2
\]

So, after distributing, we have:
\[
4a^3 + 
